# Open-source image generation (Day 1 lab)

> **日本語版 / Japanese version**: [Open in Colab](https://colab.research.google.com/github/itoksk/summer-ai-materials/blob/main/materials/notebooks/image_gen_advanced.ipynb)

The Gemini you used in the Day 1 image contest is a "finished service." Here you'll **run an open model (Stable Diffusion) with your own hands** and turn the dials directly.

This notebook lives in the public repo **github.com/itoksk/summer-ai-materials** under `materials/notebooks/`, opened from GitHub via "Open in Colab."

**Setup**
1. Menu "File -> Save a copy in Drive" (keeps your edits and images in your own Drive)
2. Menu "Runtime -> Change runtime type" -> pick **T4 GPU** (the free tier is fine)
3. Run the cells top to bottom. **Change the dials in the form on top of each cell** (open the code with "Show code").

**Rules**
- No real people's faces or names — not yourself, friends, or celebrities (portrait rights)
- No existing characters (anime / games) as-is — as you discussed in the copyright work
- No passing AI work off as human-made, no impersonation, no deepfakes (these can be crimes)
- Credit your images as "created with Stable Diffusion"; check the licence before sharing or commercial use
- Model licences are the CreativeML OpenRAIL-M / OpenRAIL++ family (they define prohibited uses)

(Based on the school class "AI image processing & copyright")

## Step 1 — Install

In [ ]:
# Upgrade diffusers etc. (the high-quality Animagine model needs a recent diffusers for long prompts)
!pip -q install --upgrade diffusers transformers accelerate safetensors

## Step 2 — Pick a model and load it

This downloads a model (a few GB) trained on billions of images. It takes a few minutes.

**Pick the model with `MODEL` in the form below.** There are two tiers:

| Tier | `MODEL` value | Notes |
|---|---|---|
| Basic (SD1.5) | `anime` / `base` / `photo` | **Fast** · **safety filter ON** · textbook dials. Start here. |
| High quality (SDXL) | `anime-xl` / `photo-xl` | **Prettier but heavy** · no built-in filter · prompts written a bit differently |

**Mind the prompt style**
- `anime-xl` (Animagine XL 4.0) … write **Danbooru-style tags** rather than natural sentences. The quality tags at the end are added for you.
- Everything else … plain natural English is fine.
- → When you pick a model, **its recommended prompt/settings are printed below** (copy them into the Step 3 form).

**About the safety filter (a real example for the reflection)**
- The basic tier (SD1.5) loads **with the NSFW filter on**. If you trip it, the image comes back **pure black** = not a bug, but the filter doing its job.
- The high-quality tier (SDXL) has **no** built-in filter.
- "If it's the same open model, why does a service (Gemini) add a filter?" — you'll feel this difference with your own hands in the final reflection.

**Weight / VRAM** — SDXL is heavy per image. If a free T4 gives `Out of memory`, go back to the basic tier or lower `steps` below.

In [ ]:
#@title ① Pick a model & load  { display-mode: "form" }
#@markdown Basic (fast, filter ON): **anime / base / photo**　|　High quality (SDXL, heavy, no filter): **anime-xl / photo-xl**
MODEL = "anime"  #@param ["anime", "base", "photo", "anime-xl", "photo-xl"]
#@markdown Change it, then press the ▶ on the left to reload.

import torch
from diffusers import (
    StableDiffusionPipeline, StableDiffusionXLPipeline,
    DPMSolverMultistepScheduler, AutoencoderKL,
)

# Check for a GPU first (otherwise to("cuda") fails): Runtime -> Change runtime type -> T4 GPU
if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU found. In the Colab menu open [Runtime] -> [Change runtime type], "
        "pick the T4 GPU accelerator, save, then run this cell again.")

# --- model table (repo name + recommended settings for each model) ---
ANIME_XL_NEG = ("lowres, bad anatomy, bad hands, text, error, missing finger, "
                "extra digits, fewer digits, cropped, worst quality, low quality, "
                "low score, bad score, average score, signature, watermark, username, blurry")
PHOTO_XL_NEG = ("worst quality, low quality, illustration, 3d, 2d, painting, cartoon, "
                "sketch, deformed iris, deformed pupils, bad anatomy, extra fingers, "
                "fused fingers, blurry, text, watermark")
SD15_PHOTO_NEG = ("cartoon, anime, sketch, drawing, 3d, render, worst quality, low quality, "
                  "bad anatomy, extra fingers, mutated hands, blurry, text, watermark")

MODELS = {
    # ---- Basic tier: SD1.5 (fast, filter ON, natural sentences OK) ----
    "anime": {"arch": "sd15", "repo": "stablediffusionapi/anything-v5",
              "prompt": "a cozy ramen shop on a rainy night, neon signs, watercolor style",
              "neg": "low quality, blurry, text, watermark", "suffix": "",
              "steps": 25, "cfg": 7.0, "w": 512, "h": 512},
    "base":  {"arch": "sd15", "repo": "stable-diffusion-v1-5/stable-diffusion-v1-5",
              "prompt": "a cozy ramen shop on a rainy night, neon signs, watercolor style",
              "neg": "low quality, blurry, text, watermark", "suffix": "",
              "steps": 25, "cfg": 7.0, "w": 512, "h": 512},
    "photo": {"arch": "sd15", "repo": "SG161222/Realistic_Vision_V5.1_noVAE", "vae": "mse",
              "prompt": "a cozy ramen shop on a rainy night, neon signs, photo, 35mm",
              "neg": SD15_PHOTO_NEG, "suffix": "",
              "steps": 25, "cfg": 7.0, "w": 512, "h": 512},
    # ---- High-quality tier: SDXL (pretty, heavy, no built-in filter) ----
    "anime-xl": {"arch": "sdxl", "repo": "cagliostrolab/animagine-xl-4.0", "lpw": True,
                 "prompt": "no humans, ramen shop interior, neon signs, rainy night, "
                           "window reflection, steam, warm lighting, detailed background",
                 "neg": ANIME_XL_NEG,
                 "suffix": ", masterpiece, high score, great score, absurdres",
                 "steps": 28, "cfg": 5.5, "w": 1024, "h": 1024},
    "photo-xl": {"arch": "sdxl", "repo": "SG161222/RealVisXL_V5.0",
                 "prompt": "a cozy ramen shop on a rainy night, neon signs reflecting on wet "
                           "pavement, cinematic lighting, photorealistic, 35mm",
                 "neg": PHOTO_XL_NEG, "suffix": "",
                 "steps": 30, "cfg": 5.0, "w": 1024, "h": 1024},
}

cfg = MODELS[MODEL]
print("loading:", MODEL, "->", cfg["repo"])

if cfg["arch"] == "sd15":
    # SD1.5: always load with the NSFW safety filter on (this runs in school)
    from transformers import CLIPImageProcessor
    from diffusers.pipelines.stable_diffusion.safety_checker import StableDiffusionSafetyChecker
    checker = StableDiffusionSafetyChecker.from_pretrained(
        "CompVis/stable-diffusion-safety-checker", torch_dtype=torch.float16)
    extractor = CLIPImageProcessor.from_pretrained("CompVis/stable-diffusion-safety-checker")
    kw = dict(torch_dtype=torch.float16, safety_checker=checker, feature_extractor=extractor)
    if cfg.get("vae") == "mse":   # *_noVAE models need a VAE added or colors look dull
        kw["vae"] = AutoencoderKL.from_pretrained("stabilityai/sd-vae-ft-mse",
                                                  torch_dtype=torch.float16)
    pipe = StableDiffusionPipeline.from_pretrained(cfg["repo"], **kw)
else:
    # SDXL: high quality. No built-in filter (see the Step 2 notes)
    kw = dict(torch_dtype=torch.float16, use_safetensors=True, add_watermarker=False)
    if cfg.get("lpw"):            # pipeline that handles long / tag / weighted prompts
        kw["custom_pipeline"] = "lpw_stable_diffusion_xl"
    pipe = StableDiffusionXLPipeline.from_pretrained(cfg["repo"], **kw)

# scheduler (how the noise is removed). The Karras variant keeps texture stable
pipe.scheduler = DPMSolverMultistepScheduler.from_config(
    pipe.scheduler.config, use_karras_sigmas=True, algorithm_type="dpmsolver++")
pipe = pipe.to("cuda")

HAS_FILTER = (cfg["arch"] == "sd15")
print("ready:", cfg["repo"], "| safety filter:", "ON" if HAS_FILTER else "none (SDXL)")
print()
print("--- recommended for this model (copy into the Step 3 form) ---")
print("prompt   :", cfg["prompt"])
print("negative :", cfg["neg"])
print(f"steps={cfg['steps']}  guidance={cfg['cfg']}  size={cfg['w']}x{cfg['h']}")
if cfg["suffix"]:
    print("quality tags", repr(cfg["suffix"]), "are appended automatically")

## Step 3 — Generate

What the dials mean:
- `prompt`: what to draw (**in English**; ask Gemini to translate if you like)
- `negative_prompt`: what to keep OUT
- `seed`: the dice. **same seed + same words = same image**
- `guidance`: how hard to obey the words (about 7 for basic / about 5 for SDXL)
- `steps`: passes from noise to image (about 25–28)

Just change the form below and press ▶. **The recommended values for your model** are printed in the Step 2 output (copy them).

In [ ]:
#@title ② Generate  { display-mode: "form" }
#@markdown In English (ask Gemini to translate). Same seed + same words = same image. For SDXL (anime-xl/photo-xl), guidance around 5 looks best.
prompt = "a cozy ramen shop on a rainy night, neon signs, watercolor style"  #@param {type:"string"}
negative_prompt = "low quality, blurry, text, watermark"  #@param {type:"string"}
seed = 42  #@param {type:"integer"}
steps = 25  #@param {type:"slider", min:10, max:40, step:1}
guidance = 7.0  #@param {type:"slider", min:1, max:15, step:0.5}

full_prompt = prompt + cfg["suffix"]          # quality tags (per model) are appended for you
generator = torch.Generator("cuda").manual_seed(seed)
image = pipe(full_prompt, negative_prompt=negative_prompt,
             num_inference_steps=steps, guidance_scale=guidance,
             width=cfg["w"], height=cfg["h"], generator=generator).images[0]
# A pure-black image = the safety filter fired (SD1.5 only). Change the words and retry
image

## Step 4 — Experiments

1. **Fix the seed, change only the words** — turn `watercolor style` into `pixel art`?
2. **Fix the words, change only the seed** — same order, a different picture
3. Try `guidance` at 2 and 15 — what happens when it obeys too hard?

(The "control a picture with words" feeling from the Day 1 contest — now at the level of the dials.)

Note: this experiment works because the dials respond cleanly on the **basic tier** and **standard SDXL**. "Fast" models (Lightning/Turbo) that draw in 4–6 steps ignore guidance and break this experiment.

In [ ]:
#@title ③ Experiment: one more image with different dials  { display-mode: "form" }
#@markdown Fix the seed and change only the words / fix the words and change only the seed / compare guidance at 2 and 15
prompt = "a cozy ramen shop on a rainy night, neon signs, pixel art"  #@param {type:"string"}
seed = 42  #@param {type:"integer"}
steps = 25  #@param {type:"slider", min:10, max:40, step:1}
guidance = 7.0  #@param {type:"slider", min:1, max:15, step:0.5}

full_prompt = prompt + cfg["suffix"]
generator = torch.Generator("cuda").manual_seed(seed)
pipe(full_prompt, negative_prompt=cfg["neg"],
     num_inference_steps=steps, guidance_scale=guidance,
     width=cfg["w"], height=cfg["h"], generator=generator).images[0]

## Reflection

- Whose pictures was this model trained on? Did they agree? — revisit the copyright question, today as the one **doing the generating**.
- How is a service like Gemini different from an open model you run yourself? (safety filter, responsibility, freedom)
  - Today you actually switched between **basic = filter ON / high-quality (SDXL) = no filter**. Why does a service add a filter?
- Is the picture you made "your work"? The AI's? The training-data artists'?